# MultiProcessing

En Python, le GIL (Global Interpreter Lock) empêche deux threads Python d’exécuter du code Python en même temps.

Résultat : le multithreading n’accélère pas les tâches CPU-intensives, comme par exemple les longs calculs, ou bien le traitement de données


Le multiprocessing, lui :

- lance plusieurs processus indépendants
- chacun a sa propre mémoire + son propre interpréteur Python
- permet d’utiliser tous les cœurs du CPU
- contourne totalement le GIL

Cela permet donc de paralleliser les taches de calculs lorsqu'on veut par exemple traiter des millions de données


Pour faire cela, Le module `multiprocessing` contient les élements suivants : 

- `Process` — créer et contrôler des processus
- `Pool` — exécuter facilement une fonction en parallèle
- `Queue`, `Pipe` — communiquer entre processus
- `Value`, `Array`, `Manager` — partager des données en toute sécurité

la documentation complete : https://docs.python.org/3/library/multiprocessing.html#the-process-class


## Créer un processus

La création d'un `Process` est tres semblable a celle d'un `Thread`:

1. On créer une instance de la classe `Process`
2. On définit une `target` et des `args`
3. On démarre le processus avec `.start()`
4. `.join()` permet d'attendre qu'un processus se termine avant d'executer le code suivant

(Et on peut créer des processus "compagnons" de notre programme, qui se terminent en meme temps que celui-ci, avec `deamon`)

In [ ]:
from multiprocessing import Process
import os
import time

def worker():
    print(f"Process {os.getpid()}")
    time.sleep(1)

p = Process(target=worker)
p.start()

In [ ]:
def worker():
    print(f"Process {os.getpid()}")
    time.sleep(1)

p = Process(target=worker)
p.start()
print("le processus est en train de travailler")
p.join()
print("Processus terminé")

### Passer des arguments

In [ ]:
def worker(n):
    print(f"Process {os.getpid()} : n = {n}")
    time.sleep(1)

p = Process(target=worker, args=(5,))
p.start()

### Terminer un processus

In [ ]:
def worker():
    print(f"Process {os.getpid()}")
    while True:
        time.sleep(1)

p = Process(target=worker)
p.start()

print("processus en cours d'execution")

In [ ]:
p.is_alive()

In [ ]:
p.terminate()

In [ ]:
p.is_alive()

## Lancer plusieurs processus

In [ ]:
def f(i):
    print(f"Process {os.getpid()}")


processes = []

for i in range(5):
    p = Process(target=f, args=(i,))
    processes.append(p)
    p.start()

for p in processes:
    p.join()


## Un premier exemple d'utilisation

On imagine un progamme devant réaliser 2 taches : 
- Calcule des factorielles
- Effectuer un calcul avec 3 loops imbriqués

In [ ]:
import math

def factorial_compute(n=10_000):
    return [math.factorial(x) for x in range(n)]

In [ ]:
def triple_nested(n=300):
    s = 0.0
    for i in range(n):
        for j in range(n):
            for k in range(n):
                s += math.sin(i) * math.cos(j) * math.tan(k+1)
    return s

In [ ]:
import time

print("début calcul 1")
start = time.perf_counter()
res1 = factorial_compute(n=10_000)
end = time.perf_counter()
print(f"fin calcul 1. Durée : {end - start}")

print("...")

print("début calcul 2")
start = time.perf_counter()
res2 = triple_nested(n=300)
end = time.perf_counter()
print(f"fin calcul 2. Durée : {end - start}")

Pour accélerer le tout, on peut créer 2 processus qui tournent chacun en parallele
- Un processus pour calculer des factorielles
- Un processus pour calculer des racines carrées

In [ ]:
p1 = Process(target=factorial_compute, args=(10_000,))
p2 = Process(target=triple_nested, args=(300,))

start = time.perf_counter()
print("début calcul 1")
p1.start()

print("début calcul 2")
p2.start()

print("...")

p1.join()
print("fin calcul 1")
p2.join()
print("fin calcul 2")

end = time.perf_counter()
print(f"Durée total: {end - start}")

Question : On observe que les résutlats ne sont pas renvoyés par nos processus... pourquoi ? 

# Partage de données entre processus

Chaque processus ayant sa propre mémoire, on ne peut pas partager directement des objets Python d'un processus a l'autre.

Cependant, il existe plusieurs méthodes nous permettant de gerer cela :

- `Value` et `Array` : qui permettent de créer un espace mémoire que peuvent se partager les différents processus
- `Manager`: qui permet de partager directement des lists, des dictionnaires, etc.
- `Queue` et `Pipe`: qui permet de structurer le transfert de données avec les files d'attentes.

## Utilisation de `Value` et `Array`

Elles permettent de créer des variables stockées sur une mémoire partagée. Il faut indiquer leur type ainsi que leur taille (pour les Arrays) avant de les utiliser.

Pour définir les types de `Value` et `Array`: 
| Typecode | Type Python | Taille                            | Notes            |
| -------- | ----------- | --------------------------------- | ---------------- |
| `'i'`    | int         | 32 bits (int32)                   | signé            |
| `'I'`    | int         | 32 bits (int32)                   | non signé        |
| `'l'`    | int         | 32 ou 64 bits selon la plateforme | signé            |
| `'L'`    | int         | 32 ou 64 bits                     | non signé        |
| `'q'`    | int         | 64 bits                           | signé            |
| `'Q'`    | int         | 64 bits                           | non signé        |
| `'f'`    | float       | 32 bits                           | simple précision |
| `'d'`    | float       | 64 bits                           | double précision |
| `'c'`    | char        | 1 byte                            | caractère unique |


In [6]:
from multiprocessing import Value, Array, Process


v = Value('i', 0)
a = Array('i', [1, 2, 3])

print(v.value, a[:])

def worker(v, a):
    v.value += 1
    a[0] = 42

p = Process(target=worker, args=(v, a))
p.start()
p.join()

print(v.value, a[:])


0 [1, 2, 3]
1 [42, 2, 3]


Utilisons cela sur le précédent exemple !

In [7]:
import math

def factorial_compute(n, a):
    for i in range(n):
        a[i] = math.factorial(i)

In [8]:
def triple_nested(n, v):
    s = 0.0
    for i in range(n):
        for j in range(n):
            for k in range(n):
                s += math.sin(i) * math.cos(j) * math.tan(k+1)
    v.value = s
 

In [13]:
N1 = 10_000
N2 = 300

v = Value('d', 0)
a = Array('L', list(range(N1)))

p1 = Process(target=factorial_compute, args=(N1, a))
p2 = Process(target=triple_nested, args=(N2, v))


p1.start()
p2.start()

p1.join()
p2.join()

In [14]:
a[:]

[1,
 1,
 2,
 6,
 24,
 120,
 720,
 5040,
 40320,
 362880,
 3628800,
 39916800,
 479001600,
 6227020800,
 87178291200,
 1307674368000,
 20922789888000,
 355687428096000,
 6402373705728000,
 121645100408832000,
 2432902008176640000,
 14197454024290336768,
 17196083355034583040,
 8128291617894825984,
 10611558092380307456,
 7034535277573963776,
 16877220553537093632,
 12963097176472289280,
 12478583540742619136,
 11390785281054474240,
 9682165104862298112,
 4999213071378415616,
 12400865694432886784,
 3400198294675128320,
 4926277576697053184,
 6399018521010896896,
 9003737871877668864,
 1096907932701818880,
 4789013295250014208,
 2304077777655037952,
 18376134811363311616,
 15551764317513711616,
 7538058755741581312,
 10541877243825618944,
 2673996885588443136,
 9649395409222631424,
 1150331055211806720,
 17172071447535812608,
 12602690238498734080,
 8789267254022766592,
 15188249005818642432,
 18284192274659147776,
 9994050523088551936,
 13175843659825807360,
 10519282829630636032,
 6711

In [12]:
v.value

177.38456529255885

## Utilisation du `Manager`

Un Manager est utile lorsque vous voulez partager des objets (comme des listes, dictionnaires, etc.) entre différents processus, car les objets normaux ne sont pas partagés par défaut.

In [19]:
import sys
sys.set_int_max_str_digits(300_000)

In [15]:
from multiprocessing import Process, Manager
import math

In [16]:
def factorial_compute(n, shared_list):
    for i in range(n):
        shared_list.append(math.factorial(i))

In [20]:
N1 = 5000
N2 = 200 

manager = Manager()

# Liste partagée pour les factorials
factorials = manager.list()

p1 = Process(target=factorial_compute, args=(N1, factorials))
p2 = Process(target=triple_nested, args=(N2, v))

p1.start()
p2.start()

p1.join()
p2.join()

In [21]:
print("Premieres factorials :", factorials[:10])
print("dernieres factorials :", factorials[-10:])
print("Résultat triple_nested :", v.value)

Premieres factorials : [1, 1, 2, 6, 24, 120, 720, 5040, 40320, 362880]
dernieres factorials : [...]
Résultat triple_nested : 198.57708412180992


## Utilisation de `Queue`
Concept identique a celui des `threads`, les `queues` Permettent de passer des messages ou des données entre processus de manière FIFO, et sont particulierement utiles pour la communication unidirectionnelle ou bidirectionnelle entre producteurs et consommateurs.

In [22]:
from multiprocessing import Process, Queue
import time
import random

def producer(queue, n_items):
    """Produit des éléments et les met dans la queue."""
    for i in range(n_items):
        item = random.randint(1, 100)
        print(f"Producteur: produit {item}")
        queue.put(item)  # ajoute l'élément dans la queue
        time.sleep(random.random())  # simule un temps de production
    queue.put(None)  # signal de fin pour le consommateur
    print("Producteur: terminé")

def consumer(queue):
    """Consomme les éléments de la queue."""
    while True:
        item = queue.get()  # récupère un élément
        if item is None:
            # Re-met le signal pour d'autres consommateurs éventuels
            queue.put(None)
            break
        print(f"Consommateur: consommé {item}")
        time.sleep(random.random())  # simule un temps de traitement
    print("Consommateur: terminé")


q = Queue()

n_items = 10
p = Process(target=producer, args=(q, n_items))
c = Process(target=consumer, args=(q,))

p.start()
c.start()

p.join()
c.join()


Producteur: produit 42
Consommateur: consommé 42
Producteur: produit 35
Consommateur: consommé 35
Producteur: produit 66
Consommateur: consommé 66
Producteur: produit 32
Consommateur: consommé 32
Producteur: produit 42
Consommateur: consommé 42
Producteur: produit 72
Consommateur: consommé 72
Producteur: produit 29
Consommateur: consommé 29
Producteur: produit 13
Consommateur: consommé 13
Producteur: produit 21
Consommateur: consommé 21
Producteur: produit 39
Consommateur: consommé 39
Producteur: terminé
Consommateur: terminé


## `Pool`: paralléliser facilement des tâches

`Pool` est la méthode la plus simple pour paralleliser l'execution d'une fonction a travers un nombre N de processus.

1. On passe `Pool(n)` dans un context manager, ou `n` désigne le nombre de processus paralleles que l'on désire créer.
2. On utilise `p.map` pour associer une fonction a des données d'executions, et cela va automatiquement répartir les données dans `n` Pools différents

In [23]:
import math

result_A = [math.factorial(x) for x in range(10_000)]

In [28]:
from multiprocessing import Pool

with Pool(os.cpu_count()) as p:
    result_B = p.map(math.factorial, list(range(10_000)))

In [25]:
assert result_A == result_B

In [27]:
import os
os.cpu_count()

12

# En résumé...

**MultiThreading** et **MultiProcessing** sont des techniques essentielles à comprendre pour être un développeur professionnel.

Cependant, dans la majorité des workflows de Data Science, Machine Learning ou Deep Learning, on voit rarement les modules `threading` ou `multiprocessing` utilisés directement. Pourquoi ?

Eh bien parce que la majorité des bibliothèques ML/DL (NumPy, pandas, PyTorch, TensorFlow) gèrent déjà la parallélisation en interne !

- Les GPUs NVIDIA (à travers CUDA, PyTorch, TensorFlow) offrent une parallélisation massive pour l’IA.
- Spark offre du traitement distribué ou sur plusieurs cœurs/machines pour les grands datasets.

## Quand multiprocessing ou threading peuvent être utiles :

- Préprocessing de données qui n’est pas vectorisable avec NumPy/pandas (par exemple du texte).
- Scraping web ou appels API massifs → threading (I/O-bound).
- Paralléliser des expériences/simulations indépendantes → multiprocessing (CPU-bound).
- Pipeline sur CPU multi-cœurs pour des jobs personnalisés, là où Spark ou Dask seraient trop lourds.

Le mot de la fin : vous en aurez rarement besoin… mais quand vous en aurez besoin, vous serez bien content de les connaître !